## New calcification evolution equation

### Comments
- We are trying to optimize the initial model changing the calcification equation. In this case c_cal will not be triggered after encapsulation, but it will start from the beginning with a very slow evolution and then grow exponentially, untill reaching a plateau.


### Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import imageio.v2 as imageio
import os
import cv2
import csv
from scipy.stats import qmc
from shapely.geometry import Polygon, Point, box
from shapely import wkt, affinity
from typing import List, Tuple, Dict
from scipy.optimize import minimize
import statistics as stat
import matplotlib as mpl
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# ------- let's import our evolution function ----
import model_alternative_cal as mfun4

### Parameters

In [ ]:
# fixed parameters
dx = 0.05 # (mm)
lx = 80 
ly = 120
dt = 0.00025 # (days)


# lesion
a = 0.14 # threshold for lesion growth (g/mm2)
eps = 1 # lesion sharpness
r0 = 1.5 # lesion initial radius (grid cells)
alfa = 1 # initial concentration of lesion, between [0,1]
count = 1 # number of lesions --ID identification--
age1 = 0.0 # (days)
k = 9 #lesion reaction coefficient (days^-1) 
D = 5e-3 # lesion diffusivity (mm^2/days)

# fibroblasts 
k_f = 7.4 #4 # fibroblasts proliferation coefficient (days^-1)
xi_kin = 9e-2 #.5e-2 # chemokinetic coefficient (mm^3/day)
xi_tax = 9e-2 # 1.05e-2 # chemotactic coefficient (mm^2/day)

# calcification
dcal = 0.001
k_nuc = 0.0035
k_grow = 2.5

### Geometric initialization

In [ ]:
# definim el nostre espai
coord = ((0., 0.), (0., ly), (lx, ly), (lx,0.), (0., 0.)) # creem el nostre grid
central_region = Polygon(coord) 
origin1 = (21,60)#central_region.centroid.coords[0]
x, y = central_region.exterior.xy 

In [ ]:
# representem el sistema
plt.plot(x, y, 'k-', linewidth=2)
plt.fill(x, y, color='orange', alpha=0.4)
plt.scatter(origin1[0], origin1[1], color = 'red')
plt.show()

### Optimization

In [ ]:
# we create a dictionary with the parameters
fixed_params = {"a": a, "eps": eps, "r0":r0, "alfa": alfa, "count": count, "dx": dx, "lx": lx, "ly": ly, "dt": dt, "age1": age1, "central_region": central_region, "origin1": origin1} 
estimated_params = {"k": k, "D": D, "xi": xi_kin, "xi_tax": xi_tax, "k_f": k_f, "dcal": dcal, "k_nuc": k_nuc, "k_grow": k_grow}
params = {**estimated_params, **fixed_params}

print(params.keys())

In [ ]:
iter = 200000
state, ages, origins, radius, radius_lc, full_radius, calcification, c_f1, max_grad, DIF_kin_vector, DIF_tax_vector, prol_vector, cal_rate_vec, phases, cfl_vec, overshoot_vec, n_cells_corrected_vec, c_l_time, c_cal_time, c_f_time  = mfun4.model_evolution_loop(iter, params)

##### Results and plots

In [ ]:
# heatmaps for the variables
cmap_cl = plt.cm.YlOrRd  
cmap_cf = plt.cm.YlGnBu 
cmap_ccal = LinearSegmentedColormap.from_list('ccal', ['#ffffe5', '#737373', '#000000'])

mpl.rcParams.update({"text.usetex": True, "font.family": "serif", "font.serif": ["Computer Modern Roman"], "axes.labelsize": 12, "font.size": 11, "legend.fontsize": 10, "xtick.labelsize": 10, "ytick.labelsize": 10})

# phases visualization
# --- colour map for each phase ---
phase_colors = {0:'#ffffff', 1:'#fff3cd', 2:'#ffd6a5', 3:'#cce5ff', 4:'#d4edda'}
phase_labels = {0: 'Disappeared', 1: 'Phase I: Growing', 2: 'Phase II: Encapsulating', 3: 'Phase III: Calcifying', 4: 'Phase IV: Resolved'}

In [ ]:
# data organization
    # fields
c_l = state["c_l_1"].T
c_f = c_f1.T
c_cal = calcification["c_cal_1"].T

    # radius evolution variables
rad_les = radius["r_1"]
rad_calc = radius_lc["r_lc_1"]
full_radiuss = full_radius["full_r_1"]

time = np.arange(iter) * dt
extent_full = [0, c_l.shape[1]*dx, 0, c_l.shape[0]*dx]

In [ ]:
df = pd.DataFrame({
    "rad_les": rad_les,
    "rad_calc": rad_calc,
    "full_radiuss": full_radiuss
})

#df.to_csv(f"radius_new_cal.csv", index=False)

In [ ]:
fig = plt.figure(figsize=(6, 10))
outer = gridspec.GridSpec(3, 1, figure=fig, hspace=0.3)

rows = [([c_l_time['c_l_79999'].T,   c_l_time['c_l_103999'].T,   c_l_time['c_l_149999'].T], cmap_cl,  r'$c_l$'),
    ([c_f_time['c_f_79999'].T,   c_f_time['c_f_103999'].T,   c_f_time['c_f_149999'].T],  cmap_cf,   r'$c_f$'),
    ([c_cal_time['c_cal_79999'].T,   c_cal_time['c_cal_103999'].T,   c_cal_time['c_cal_149999'].T], cmap_ccal, r'$c_{cal}$')]
    # day 20                      # day 27                        # day 40
letters = ['(a)', '(b)', '(c)']
time_labels = ['Day 20', 'Day 26', 'Day 40']

for i, (letter, (data_list, cmap, clabel)) in enumerate(zip(letters, rows)):

    # 4 columns: 3 plots + 1 narrow colorbar column
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 4,
        subplot_spec=outer[i],
        wspace=0.05,
        width_ratios=[1, 1, 1, 0.05]  # ← colorbar gets its own thin column
    )

    for j, data in enumerate(data_list):
        ax = fig.add_subplot(inner[j])
        im = ax.imshow(data, cmap=cmap, origin='lower', extent=extent_full,
                       interpolation='bilinear', vmin=0, vmax=1)
        ax.set_xlabel(r'$x$ (mm)')

        if j == 0:
            ax.set_ylabel(r'$y$ (mm)')
            ax.set_title(letter, loc='left', fontweight='bold')
        else:
            ax.tick_params(labelleft=False)

        if i == 0:
            ax.set_title(time_labels[j], fontsize=10)

    # colorbar in the dedicated 4th column — same size as the plots
    cax = fig.add_subplot(inner[3])
    plt.colorbar(im, cax=cax, label=clabel)

#plt.savefig("new_cal_outputs.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

e = 1
phase_array = phases[f'phase_{e}']
current_phase = phase_array[0]
start_idx = 0

for idx in range(1, iter):
    if phase_array[idx] != current_phase or idx == iter - 1:
        ax.axvspan(time[start_idx], time[idx], color=phase_colors.get(int(current_phase), 'white'), alpha=1.0, lw=0)
        start_idx = idx
        current_phase = phase_array[idx]

ax.plot(time, rad_les, linewidth=1.0, color='#bd0026', label=r'$r_{lesion}(t)$')
ax.plot(time, rad_calc, linewidth=1.0, color='#495057', label=r'$r_{cal}(t)$')
ax.plot(time, full_radiuss, linewidth=1.0, color='#005f73', label=r'$r_{granuloma}(t)$')

phase_patches = [Patch(facecolor=phase_colors[p], alpha=1.0, label=phase_labels[p]) for p in sorted(phase_colors.keys()) if p in np.unique(phases['phase_1'])]

radius_handles, _ = ax.get_legend_handles_labels()
ax.legend(handles=radius_handles + phase_patches, loc='upper left', fontsize=9)


ax.set_xlabel(r'$t$ (days)')
ax.set_ylabel(r'$r(t)$ (mm)')
ax.set_xlim(time[0], time[-1])
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
# plt.savefig('new_cal_radius_phases.pdf', dpi=300, bbox_inches='tight')
plt.show()

##### comparisons with original model

In [ ]:
time = np.arange(iter) * dt

df_orig = pd.read_csv('radius_baseline.csv')
df_new  = pd.read_csv('radius_new_cal.csv')
fig, ax = plt.subplots(figsize=(8, 4))


ax.plot(time, df_orig['rad_les'], linewidth=1.0, color='#bd0026',  ls='--', label=r'$r_{l}(t)$ original')
ax.plot(time, df_orig['rad_calc'], linewidth=1.0, color='#495057',  ls='--', label=r'$r_{cal}(t)$ original')
ax.plot(time, df_orig['full_radius'], linewidth=1.0, color='#005f73',  ls='--', label=r'$r_{granuloma}(t)$ original')

ax.plot(time,  df_new['rad_les'],    color='#bd0026',  lw=1.0, label=r'$r_l$ new')
ax.plot(time,  df_new['rad_calc'],  color='#495057', lw=1.0, label=r'$r_{cal}$ new')
ax.plot(time, df_new['full_radiuss'], color = '#005f73', lw = 1.0, label = r'$r_{granuloma}$ new')

ax.set_xlabel('Time (days)')
ax.set_ylabel('Radius (mm)')
ax.grid(True, alpha = 0.3)
ax.legend()
plt.tight_layout()
plt.savefig('cal_comparison.pdf')

In [ ]:
## radiuses comparison
rl_28 = df_orig['rad_les'][int(28/dt)-1]
rl_28_new = df_new['rad_les'][int(28/dt)-1]
error_les = np.abs(rl_28-rl_28_new)/rl_28*100

r_crown = df_orig['full_radius'][int(28/dt)-1] - df_orig['rad_les'][int(28/dt)-1]
r_crown_new = df_new['full_radiuss'][int(28/dt)-1] - df_new['rad_les'][int(28/dt)-1]
error_crown = np.abs(r_crown - r_crown_new)/r_crown*100

print(f'At day 28, lesion radius for the new model is {rl_28_new}, corresponding to a relative difference of {error_les}')
print(f'At day 28, fibroblast crown thickness for the new model is {r_crown_new}, corresponding to a relative difference of {error_crown}')

In [ ]:
def get_phase_transitions(phase_array, time):
    transitions = []

    for idx in range(1, len(phase_array)):
        if phase_array[idx] != phase_array[idx - 1]:
            transitions.append({
                'time': time[idx],
                'from': phase_array[idx - 1],
                'to': phase_array[idx]
            })

    return transitions

transitions = get_phase_transitions(phases['phase_1'], time)
pd.DataFrame(transitions).to_csv("phases_new.csv", index=False)

In [ ]:
## phases comparison
phases_orig = pd.read_csv('phases_original.csv')
phases_new  = pd.read_csv('phases_new.csv')

comparison = pd.merge(
    phases_orig,
    phases_new,
    on=['from', 'to'],
    suffixes=('_orig', '_new')
)

comparison['diff_days'] = (
    comparison['time_new']
    - comparison['time_orig']
)

comparison['diff_pct'] = (
    comparison['diff_days'].abs()
    / comparison['time_orig']
    * 100
)

print(comparison)